# Various model pipelines

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Use project root relative to this notebook
BASE_DIR = Path('/home/amraas/projects/realestatecons')
DATA_DIR = BASE_DIR / 'data' / 'raw'

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

In [2]:
import sys

sys.path.append(str(BASE_DIR))

from src.data.preprocess import clean_test_data, clean_train_data
from src.features.features import add_engineered_features,drop_highly_correlated_features


In [3]:
train_df_cleaned = clean_train_data(train_df)
test_df_cleaned = clean_test_data(test_df, train_df)

train_df_features = add_engineered_features(train_df_cleaned)
test_df_features = add_engineered_features(test_df_cleaned)

In [5]:
train_df = drop_highly_correlated_features(train_df_features)
test_df = drop_highly_correlated_features(test_df_features)

In [7]:
train_df.columns

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearRemodAdd', 'RoofStyle', 'RoofMatl',
       'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual',
       'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF',
       'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical',
       '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'BsmtFullBath', 'BsmtHalfBath',
       'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageQual', 'GarageCond',
       'PavedDrive', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3Ss

## Splitting

In [8]:
train_df.shape

(1460, 83)

In [9]:
test_df.shape

(1459, 82)

In [12]:
from sklearn.model_selection import train_test_split
X_train = train_df.drop(columns='SalePrice')
y_train = train_df['SalePrice']

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=42)

In [14]:
X_train.shape, X_val.shape

((1022, 82), (438, 82))

## Linear Regression

In [18]:
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SUBMISSIONS_DIR = BASE_DIR / "reports" / "figures" / "submissions"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

X_full = train_df.drop(columns="SalePrice")
y_full = train_df["SalePrice"]


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = [c for c in X.columns if c not in numeric_features]

    numeric_transformer = Pipeline(
        steps=[("scaler", StandardScaler(with_mean=False))]
    )
    categorical_transformer = OneHotEncoder(handle_unknown="ignore")

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )


def save_submission(preds: np.ndarray, name: str) -> Path:
    submission = pd.read_csv(BASE_DIR / "sample_submission.csv")
    submission["SalePrice"] = preds
    out_path = SUBMISSIONS_DIR / f"{name}.csv"
    submission.to_csv(out_path, index=False)
    print(f"Saved submission: {out_path}")
    return out_path


def run_grid_search(
    model,
    param_grid: dict,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    model_name: str,
) -> Pipeline:
    preprocessor = build_preprocessor(X_train)
    pipeline = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    grid = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
    )
    grid.fit(X_train, y_train)

    scoring = {
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2",
    }
    cv_results = cross_validate(
        grid.best_estimator_, X_train, y_train, cv=cv, scoring=scoring
    )

    print(f"{model_name} best params: {grid.best_params_}")
    print(
        f"{model_name} CV mean RMSE: {-cv_results['test_rmse'].mean():.4f}"
    )
    print(
        f"{model_name} CV mean MAE: {-cv_results['test_mae'].mean():.4f}"
    )
    print(f"{model_name} CV mean R2: {cv_results['test_r2'].mean():.4f}")

    return grid.best_estimator_


# Linear Regression via SGDRegressor
linear_model = SGDRegressor(
    penalty=None, random_state=42, learning_rate="invscaling"
)
linear_param_grid = {
    "model__eta0": [0.001, 0.01, 0.1],
    "model__max_iter": [1000, 2000],
}

best_linear = run_grid_search(
    linear_model, linear_param_grid, X_train, y_train, "LinearRegression"
)
best_linear.fit(X_full, y_full)
linear_preds = best_linear.predict(test_df)
save_submission(linear_preds, "linear_regression")

LinearRegression best params: {'model__eta0': 0.001, 'model__max_iter': 1000}
LinearRegression CV mean RMSE: 61772392588365.1484
LinearRegression CV mean MAE: 61770273093498.2031
LinearRegression CV mean R2: -1002889026854500352.0000
Saved submission: /home/amraas/projects/realestatecons/reports/figures/submissions/linear_regression.csv


PosixPath('/home/amraas/projects/realestatecons/reports/figures/submissions/linear_regression.csv')

## Lasso Regression

In [25]:
# Lasso Regression via SGDRegressor
lasso_model = SGDRegressor(
    penalty="l1", random_state=42, learning_rate="invscaling"
)
lasso_param_grid = {
    "model__eta0": [0.001, 0.01, 0.1],
    "model__max_iter": [1000, 2000,10000],
    "model__alpha": [1e-4, 1e-3, 1e-2, 1, 5],
}

best_lasso = run_grid_search(
    lasso_model, lasso_param_grid, X_train, y_train, "Lasso"
)
best_lasso.fit(X_full, y_full)
lasso_preds = best_lasso.predict(test_df)
save_submission(lasso_preds, "lasso")

Lasso best params: {'model__alpha': 1, 'model__eta0': 0.001, 'model__max_iter': 1000}
Lasso CV mean RMSE: 61489912138182.2266
Lasso CV mean MAE: 61487790419475.2031
Lasso CV mean R2: -996721917423291136.0000
Saved submission: /home/amraas/projects/realestatecons/reports/figures/submissions/lasso.csv


PosixPath('/home/amraas/projects/realestatecons/reports/figures/submissions/lasso.csv')

## Ridge Regression

In [20]:
# Ridge Regression via SGDRegressor
ridge_model = SGDRegressor(
    penalty="l2", random_state=42, learning_rate="invscaling"
)
ridge_param_grid = {
    "model__eta0": [0.001, 0.01, 0.1],
    "model__max_iter": [1000, 2000],
    "model__alpha": [1e-4, 1e-3, 1e-2],
}

best_ridge = run_grid_search(
    ridge_model, ridge_param_grid, X_train, y_train, "Ridge"
)
best_ridge.fit(X_full, y_full)
ridge_preds = best_ridge.predict(test_df)
save_submission(ridge_preds, "ridge")

Ridge best params: {'model__alpha': 0.01, 'model__eta0': 0.001, 'model__max_iter': 1000}
Ridge CV mean RMSE: 37485602811476.7344
Ridge CV mean MAE: 37483811528818.9844
Ridge CV mean R2: -316939598981397888.0000
Saved submission: /home/amraas/projects/realestatecons/reports/figures/submissions/ridge.csv


PosixPath('/home/amraas/projects/realestatecons/reports/figures/submissions/ridge.csv')

## Random Forest

In [21]:
# Random Forest Regressor
rf_model = RandomForestRegressor(random_state=42)
rf_param_grid = {
    "model__n_estimators": [200, 500],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"],
}

best_rf = run_grid_search(
    rf_model, rf_param_grid, X_train, y_train, "RandomForest"
)
best_rf.fit(X_full, y_full)
rf_preds = best_rf.predict(test_df)
save_submission(rf_preds, "random_forest")

RandomForest best params: {'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
RandomForest CV mean RMSE: 32045.9793
RandomForest CV mean MAE: 18313.7535
RandomForest CV mean R2: 0.8252
Saved submission: /home/amraas/projects/realestatecons/reports/figures/submissions/random_forest.csv


PosixPath('/home/amraas/projects/realestatecons/reports/figures/submissions/random_forest.csv')

## Gradient Boosting (XGBoost)

In [22]:
try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError("xgboost is required for this section. Install it first.") from exc

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    tree_method="hist",
)

xgb_param_grid = {
    "model__n_estimators": [300, 600],
    "model__learning_rate": [0.03, 0.1],
    "model__max_depth": [3, 6],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
}

best_xgb = run_grid_search(
    xgb_model, xgb_param_grid, X_train, y_train, "XGBoost"
)
best_xgb.fit(X_full, y_full)
xgb_preds = best_xgb.predict(test_df)
save_submission(xgb_preds, "xgboost")

XGBoost best params: {'model__colsample_bytree': 1.0, 'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 600, 'model__subsample': 0.8}
XGBoost CV mean RMSE: 28098.1437
XGBoost CV mean MAE: 16117.4680
XGBoost CV mean R2: 0.8630
Saved submission: /home/amraas/projects/realestatecons/reports/figures/submissions/xgboost.csv


PosixPath('/home/amraas/projects/realestatecons/reports/figures/submissions/xgboost.csv')